# Clase 092 — PCA: proyección, varianza explicada, Incremental, Randomized, Kernel

PCA como reducción de dimensionalidad lineal vía SVD: elegir componentes con `explained_variance_ratio_` y aplicar las variantes **Incremental**, **Randomized** y **Kernel** según tamaño y geometría.

Requiere: `numpy`, `scikit-learn`, `matplotlib`. Sin internet: usamos `load_digits` (no MNIST) y `make_swiss_roll`.

## 1. PCA básico y componentes principales

Estandarizamos y proyectamos. `components_` son combinaciones lineales ortonormales de las features originales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

np.random.seed(42)
X = load_digits().data
y = load_digits().target
Xs = StandardScaler().fit_transform(X)

pca = PCA(random_state=42).fit(Xs)
print("X:", Xs.shape, "| components_:", pca.components_.shape)
print("primeros 5 explained_variance_ratio_:", pca.explained_variance_ratio_[:5].round(4))
assert pca.explained_variance_ratio_[0] >= pca.explained_variance_ratio_[1]
print("los ratios vienen ordenados de mayor a menor: OK")

## 2. Varianza explicada y elección de `n_components`

Curva de varianza acumulada. El menor `d` con `cumsum >= 0.95` es la regla práctica.

In [ ]:
cum = np.cumsum(pca.explained_variance_ratio_)
n95 = int(np.searchsorted(cum, 0.95) + 1)
print(f"componentes para 95% varianza: {n95} (de {X.shape[1]})")

# usar el ratio como n_components lo hace automatico
pca95 = PCA(n_components=0.95, random_state=42).fit(Xs)
print("PCA(n_components=0.95) -> n_components_ =", pca95.n_components_)
assert pca95.n_components_ == n95

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(cum) + 1), cum, color="#37a")
plt.axhline(0.95, ls="--", color="#c33", lw=0.8, label="95%")
plt.axvline(n95, ls=":", color="#3a7", lw=0.8, label=f"n={n95}")
plt.xlabel("n componentes"); plt.ylabel("varianza acumulada")
plt.title("Scree acumulado: codo cerca del 95%")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Proyección 2D

Los dos primeros componentes ya separan bastante los dígitos.

In [ ]:
proj = PCA(n_components=2, random_state=42).fit_transform(Xs)
plt.figure(figsize=(7, 5))
sc = plt.scatter(proj[:, 0], proj[:, 1], c=y, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(sc, label="digito")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("digits proyectado a 2D con PCA")
plt.tight_layout(); plt.show()
print("proyeccion:", proj.shape)

## 4. IncrementalPCA (mini-batches)

`partial_fit` procesa el dataset por lotes, útil cuando `X` no entra en RAM. El resultado aproxima al PCA full: verificamos con similitud coseno entre componentes.

In [ ]:
from sklearn.decomposition import IncrementalPCA

n_comp = 10
ipca = IncrementalPCA(n_components=n_comp)
for batch in np.array_split(Xs, 20):
    ipca.partial_fit(batch)

pca_full = PCA(n_components=n_comp, svd_solver="full", random_state=42).fit(Xs)

cos = np.abs(np.sum(ipca.components_ * pca_full.components_, axis=1))
cos /= (np.linalg.norm(ipca.components_, axis=1) * np.linalg.norm(pca_full.components_, axis=1))
print("similitud coseno |componente| (primeros 5):", cos[:5].round(4))
assert cos[0] > 0.99, "el primer componente debe coincidir con el PCA full"
print("IncrementalPCA aproxima al PCA full: OK")

## 5. Randomized vs Full: velocidad

`svd_solver="randomized"` encuentra los primeros componentes mucho más rápido que `"full"` cuando `n_components << d`.

In [ ]:
import time

def cronometrar(solver, n_comp=10, reps=5):
    t0 = time.perf_counter()
    for _ in range(reps):
        PCA(n_components=n_comp, svd_solver=solver, random_state=42).fit(Xs)
    return (time.perf_counter() - t0) / reps

t_full = cronometrar("full")
t_rand = cronometrar("randomized")
print(f"full:       {t_full*1000:.2f} ms")
print(f"randomized: {t_rand*1000:.2f} ms")
print("ambos recuperan el mismo primer componente (signo aparte).")

## 6. KernelPCA sobre swiss roll

PCA lineal aplasta el rollo; `KernelPCA` con kernel RBF captura la estructura no lineal.

In [ ]:
from sklearn.datasets import make_swiss_roll
from sklearn.decomposition import KernelPCA

Xsr, color = make_swiss_roll(n_samples=1000, noise=0.05, random_state=42)

lin = PCA(n_components=2, random_state=42).fit_transform(Xsr)
kpca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04, random_state=42).fit_transform(Xsr)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(lin[:, 0], lin[:, 1], c=color, cmap="Spectral", s=8)
ax[0].set_title("PCA lineal (aplasta)")
ax[1].scatter(kpca[:, 0], kpca[:, 1], c=color, cmap="Spectral", s=8)
ax[1].set_title("KernelPCA RBF")
for a in ax:
    a.set_xlabel("comp 1"); a.set_ylabel("comp 2")
plt.tight_layout(); plt.show()
assert kpca.shape == (1000, 2)
print("KernelPCA proyecta el swiss roll a 2D: OK")

## Ejercicios

1. Aplicá `StandardScaler` + `PCA(n_components=0.95)` sobre `load_digits` y reportá cuántos componentes quedaron.
2. Graficá la curva de varianza acumulada y marcá el codo (hecho en la sección 2 — probá con `load_wine`).
3. Compará tiempos de `PCA(svd_solver="full")` vs `"randomized"` variando `n_components`.
4. Verificá que `IncrementalPCA` se aproxima al PCA full con similitud coseno > 0.99.
5. Sobre `make_swiss_roll`, compará `KernelPCA(kernel="rbf")` contra PCA lineal en un scatter 2D.

## Conclusiones

- PCA proyecta al subespacio de máxima varianza; `explained_variance_ratio_` guía cuántos componentes conservar.
- **Escalá siempre** antes (`StandardScaler`): PCA es sensible a la escala.
- Usá `IncrementalPCA` si `X` no entra en RAM y `randomized` si querés velocidad con `n_components` chico.
- `KernelPCA` captura estructura no lineal donde el PCA lineal falla.

## ✅ Soluciones de los ejercicios

Cinco ejercicios de PCA. El README usa MNIST; para correr **offline y en < 60s** usamos `load_digits` (8x8) como sustituto —la mecánica es idéntica— y un dataset sintético de alta dimensión donde el solver *randomized* se nota. `n_jobs=1`.

**Ejercicio 1 — PCA(0.95).** `StandardScaler` + `PCA(n_components=0.95)`; cuántos componentes quedan.

In [ ]:
import numpy as np, time, matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_swiss_roll
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, IncrementalPCA, KernelPCA

X, y = load_digits(return_X_y=True)
Xs = StandardScaler().fit_transform(X)
pca95 = PCA(n_components=0.95, random_state=42).fit(Xs)
print(f'componentes para 95% varianza: {pca95.n_components_} (de {X.shape[1]})')

**Ejercicio 2 — Curva de varianza acumulada.** Marcamos dónde se alcanza el 95%.

In [ ]:
full = PCA(random_state=42).fit(Xs)
cum = np.cumsum(full.explained_variance_ratio_)
k95 = int(np.argmax(cum >= 0.95)) + 1
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cum) + 1), cum, '-')
plt.axhline(0.95, color='r', ls='--'); plt.axvline(k95, color='g', ls='--')
plt.xlabel('n componentes'); plt.ylabel('varianza acumulada')
plt.title(f'Codo en 95% -> {k95} componentes'); plt.tight_layout(); plt.show()
print('codo (95%):', k95, 'componentes')

**Ejercicio 3 — `full` vs `randomized`.** En un dataset de alta dimensión, el solver randomized es más rápido para pocos componentes.

In [ ]:
rng = np.random.default_rng(42)
# dataset alto-dimensional low-rank: 2000 x 400
base = rng.normal(size=(2000, 20)) @ rng.normal(size=(20, 400))
Xhd = base + rng.normal(0, 0.1, base.shape)
def timeit_pca(solver, reps=3):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        PCA(n_components=10, svd_solver=solver, random_state=42).fit(Xhd)
        ts.append(time.perf_counter() - t0)
    return min(ts)
t_full = timeit_pca('full'); t_rand = timeit_pca('randomized')
print(f'full       : {t_full*1000:.1f} ms')
print(f'randomized : {t_rand*1000:.1f} ms')
print('randomized aproxima los top-k componentes sin descomponer toda la matriz.')

**Ejercicio 4 — IncrementalPCA por lotes.** Verificamos que se aproxima al PCA full (similitud coseno entre componentes > 0.99).

In [ ]:
k = 10
pca_full = PCA(n_components=k, svd_solver='full', random_state=42).fit(Xs)
ipca = IncrementalPCA(n_components=k)
for chunk in np.array_split(Xs, 100):
    ipca.partial_fit(chunk)
cos = np.abs(np.sum(pca_full.components_ * ipca.components_, axis=1) /
             (np.linalg.norm(pca_full.components_, axis=1) *
              np.linalg.norm(ipca.components_, axis=1)))
print('cosine sim por componente (primeros 5):', np.round(cos[:5], 4))
assert cos[:3].min() > 0.99, 'los primeros componentes deben coincidir con el PCA full'
print('OK: IncrementalPCA reproduce el PCA full procesando por lotes (out-of-core)')

**Ejercicio 5 — KernelPCA sobre Swiss roll.** El kernel RBF desenrolla la variedad; el PCA lineal solo la aplasta.

In [ ]:
Xsr, t = make_swiss_roll(n_samples=1000, random_state=42)
lin = PCA(n_components=2).fit_transform(Xsr)
kpca = KernelPCA(kernel='rbf', gamma=0.04, n_components=2).fit_transform(Xsr)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(lin[:, 0], lin[:, 1], c=t, cmap='viridis', s=8); ax[0].set_title('PCA lineal')
ax[1].scatter(kpca[:, 0], kpca[:, 1], c=t, cmap='viridis', s=8); ax[1].set_title('KernelPCA RBF')
plt.tight_layout(); plt.show()
print('El gradiente de color queda ordenado con KernelPCA: capturo la estructura no lineal.')